# IGBT Time-Series Degradation and Breakdown Prediction
### Smart India Hackathon (SIH) Predictive Maintenance Pipeline

**Project Purpose:**
Predict future IGBT leakage current from chronological electrical measurements and provide signals suitable for predictive-maintenance / anomaly monitoring.

**Architecture Principles:**
1. **Strict Chronological Sequence:** No data shuffling. Future observations never enter the training period.
2. **Leakage-Safe Feature Engineering:** Rolling windows are shifted by 1 observation before computing rolling statistics.
3. **ML Pipeline:** Median Imputation -> StandardScaler -> GradientBoostingRegressor (n=300, lr=0.03, depth=3).
4. **Validation:** 5-Fold TimeSeriesSplit cross-validation.
5. **SIH Monitoring Logic:** Anomaly detection via prediction residuals and dynamic drift thresholds.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11
print("Environment configured successfully.")


## 1. Data Ingestion & Chronological Validation
Load sequential IGBT time-series data (Breakdown_timeseries_microampere.csv).


In [ ]:
DATA_FILE = "data/Breakdown_timeseries_microampere.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "../data/Breakdown_timeseries_microampere.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "data/Breakdown_timeseries_microampere.csv"

df_raw = pd.read_csv(DATA_FILE)
print(f"Loaded dataset: {DATA_FILE}")
print(f"Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

# Sort chronologically by time_minutes
df = df_raw.sort_values("time_minutes").reset_index(drop=True)

# Validate duplicates and missing data
print(f"Duplicate timestamps: {df['time_minutes'].duplicated().sum()}")
print(f"Missing values per column:
{df.isnull().sum()}")
df.head(10)


## 2. Leakage-Free Time-Series Feature Engineering
- **Lags:** 1, 2, 3, 6, 12 on Vce and Ic_microampere
- **Rolling Stats:** Windows 3, 6, 12, 24 with mean and std (shifted by 1)
- **Deltas:** voltage_delta, current_delta
- **Interaction:** voltage_current_product
- **Warmup:** Drop initial incomplete history rows


In [ ]:
def engineer_timeseries_features(data: pd.DataFrame) -> pd.DataFrame:
    df_feat = data.copy()
    
    # Historical Lag Features
    lags = [1, 2, 3, 6, 12]
    for lag in lags:
        df_feat[f"vce_lag_{lag}"] = df_feat["Collector_Emitter_Voltage_Vce"].shift(lag)
        df_feat[f"ic_lag_{lag}"] = df_feat["Leakage_Current_Ic_microampere"].shift(lag)
        
    # Leakage-Safe Rolling Statistics
    windows = [3, 6, 12, 24]
    vce_shifted = df_feat["Collector_Emitter_Voltage_Vce"].shift(1)
    ic_shifted = df_feat["Leakage_Current_Ic_microampere"].shift(1)
    for w in windows:
        df_feat[f"vce_roll_mean_{w}"] = vce_shifted.rolling(window=w).mean()
        df_feat[f"vce_roll_std_{w}"] = vce_shifted.rolling(window=w).std()
        df_feat[f"ic_roll_mean_{w}"] = ic_shifted.rolling(window=w).mean()
        df_feat[f"ic_roll_std_{w}"] = ic_shifted.rolling(window=w).std()
        
    # Change & Interaction Features
    df_feat["voltage_delta"] = df_feat["Collector_Emitter_Voltage_Vce"].diff(1)
    df_feat["current_delta"] = df_feat["Leakage_Current_Ic_microampere"].diff(1)
    df_feat["voltage_current_product"] = df_feat["Collector_Emitter_Voltage_Vce"] * df_feat["Leakage_Current_Ic_microampere"]
    
    # Warmup removal
    return df_feat.dropna().reset_index(drop=True)

df_features = engineer_timeseries_features(df)
print(f"Feature matrix clean shape: {df_features.shape}")
df_features.head()


## 3. Chronological Train-Test Split (80% Past / 20% Future)


In [ ]:
feature_cols = [c for c in df_features.columns if c not in ["time_minutes", "Leakage_Current_Ic_microampere"]]
target_col = "Leakage_Current_Ic_microampere"

X = df_features[feature_cols]
y = df_features[target_col]
time_idx = df_features["time_minutes"]

split_idx = int(0.8 * len(df_features))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
time_train, time_test = time_idx.iloc[:split_idx], time_idx.iloc[split_idx:]

print(f"Training instances: {len(X_train)} (Past observations)")
print(f"Testing instances:  {len(X_test)} (Future observations)")


## 4. Pipeline & TimeSeries Cross-Validation


In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    ))
])

tscv = TimeSeriesSplit(n_splits=5)
cv_maes, cv_r2s = [], []

print("Running 5-Fold TimeSeriesSplit...")
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    pipeline.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    y_val_pred = pipeline.predict(X_train.iloc[val_idx])
    mae = mean_absolute_error(y_train.iloc[val_idx], y_val_pred)
    r2 = r2_score(y_train.iloc[val_idx], y_val_pred)
    cv_maes.append(mae)
    cv_r2s.append(r2)
    print(f"  Fold {fold}: MAE = {mae:.4f} uA, R2 = {r2:.4f}")

print(f"
Cross-Validation MAE: {np.mean(cv_maes):.4f} uA")
print(f"Cross-Validation R2:  {np.mean(cv_r2s):.4f}")

pipeline.fit(X_train, y_train)
print("Final model trained on all historical training data.")


## 5. Evaluation Metrics & Required Plots
Required Plots:
1. actual_vs_predicted_leakage_current_over_time
2. prediction_error_over_time
3. top_feature_importance


In [ ]:
y_pred = pipeline.predict(X_test)
residuals = y_test.values - y_pred

test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)

print("=" * 50)
print("FUTURE TEST EVALUATION (Unseen Degradation Period)")
print("=" * 50)
print(f"MAE:  {test_mae:.4f} uA")
print(f"RMSE: {test_rmse:.4f} uA")
print(f"R2:   {test_r2:.4f}")
print("=" * 50)

# 1. Actual vs Predicted Over Time
plt.figure(figsize=(15, 6))
plt.plot(time_train, y_train, label="Training (Observed)", color="#1E40AF", alpha=0.7)
plt.plot(time_test, y_test, label="Future Ground Truth (Actual)", color="#0F172A", linewidth=2)
plt.plot(time_test, y_pred, label="GBR Prediction (Forecast)", color="#EA580C", linestyle="--", linewidth=2)
plt.axvline(x=time_test.iloc[0], color="#DC2626", linestyle=":", label="Split Boundary")
plt.title("Actual vs Predicted IGBT Leakage Current Over Time", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Time (Minutes)", fontsize=12)
plt.ylabel("Leakage Current Ic (uA)", fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig("actual_vs_predicted_leakage_current_over_time.png", dpi=300)
plt.show()

# 2. Prediction Error Over Time
plt.figure(figsize=(15, 5))
plt.plot(time_test, residuals, label="Prediction Residual (Actual - Pred)", color="#DC2626", linewidth=1.5)
plt.axhline(0, color="#64748B", linestyle="--")
plt.fill_between(time_test, -3*test_mae, 3*test_mae, color="#E2E8F0", alpha=0.6, label="+/- 3-Sigma Band")
plt.title("Prediction Residual Error Over Time (Degradation Tracking)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Time (Minutes)", fontsize=12)
plt.ylabel("Residual Error (uA)", fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig("prediction_error_over_time.png", dpi=300)
plt.show()

# 3. Top Feature Importance
gbr = pipeline.named_steps["regressor"]
imp = gbr.feature_importances_
top_indices = np.argsort(imp)[-12:]

plt.figure(figsize=(12, 6))
plt.barh(np.array(feature_cols)[top_indices], imp[top_indices], color="#1E40AF", edgecolor="#1E3A8A")
plt.title("Top 12 Time-Series Feature Importances", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Relative Importance Score", fontsize=12)
plt.tight_layout()
plt.savefig("top_feature_importance.png", dpi=300)
plt.show()


## 6. SIH Predictive Maintenance & Early Warning Monitoring Logic


In [ ]:
def sih_monitor(actual_uA: float, predicted_uA: float, healthy_sigma: float) -> dict:
    res = actual_uA - predicted_uA
    abs_res = abs(res)
    drift_pct = (res / (predicted_uA + 1e-9)) * 100.0
    
    if abs_res <= 2.0 * healthy_sigma:
        verdict = "PASS"
        action = "Normal degradation rate. Device healthy."
    elif abs_res <= 4.0 * healthy_sigma:
        verdict = "HOLD"
        action = "Elevated drift rate. Charge trapping or localized heating detected. Re-test scheduled."
    else:
        verdict = "REJECT"
        action = "Severe anomaly. Breakdown onset or threshold collapse. Immediate replacement required."
        
    return {
        "actual_uA": actual_uA,
        "predicted_uA": predicted_uA,
        "residual_uA": res,
        "drift_pct": drift_pct,
        "verdict": verdict,
        "action": action
    }

print("SIH Monitor Module initialized.")
